# CWS reviewer robustness tests — station holdout, uncertainty, and station diagnostics

This notebook is the reviewer-facing robustness companion to the existing project pipeline.

It keeps the notebook user-facing and delegates heavy work to `qc_cws_reviewer_robustness_tests.py`, which itself reuses the audited scripts:

- `qc_cws_residual_risk_model_audited.py`
- `qc_risk_fusion_audited.py`

The notebook answers likely reviewer questions:

1. **Cold-start generalization:** does the residual-risk model work on CWS stations that were never seen during training?
2. **Station-history dependence:** how different are `context_history` and `context_static_met` under station holdout?
3. **Uncertainty:** are policy improvements stable across stations, or dominated by a small number of stations?
4. **Failure modes:** which stations dominate error, rescue candidates, bias-correction candidates, or transient rejects?
5. **Seasonal/reference-support shift:** do risk/event rates change across time, hour, or OWS support classes?

Edit only the user-configuration cells. All major functions live in scripts.


Compatibility note: this notebook uses a reviewer-script shim that
- drops unsupported `FusionConfig` kwargs when the local `qc_risk_fusion_audited.py` is older, and
- back-fills missing reviewer columns such as `recommended_bias_correctable` from `fusion_category`, `recommended_action`, and correction-preview fields.


In [ ]:
import os
import sys
import json
from pathlib import Path
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 220)
pd.set_option("display.max_rows", 160)
pd.set_option("display.width", 260)


## 1. Project setup

This cell imports `config_tool.py`, reads the configured project id, imports `config_project.py` from the external data folder, and derives the standard results and QC directories.


In [ ]:
USE_CONFIG_FILES = True
city = os.environ.get("QC_PROJECT_ID", "project_id")
PROJECT_DIR = Path(os.environ.get("QC_DATA_ROOT", Path.cwd() / "data")).expanduser().resolve() / city

if USE_CONFIG_FILES:
    try:
        cwd = os.getcwd()
        cwd_main = os.path.abspath(os.path.join(cwd, os.pardir))
        os.chdir(cwd_main)
        print("cwd_main:", cwd_main)

        import config_tool as cfm

        cwd_project = Path(cfm.cwd_data) / city
        os.chdir(cwd_project)
        print("cwd_project:", cwd_project)

        if str(cwd_project) not in sys.path:
            sys.path.insert(0, str(cwd_project))
        sys.modules.pop("config_project", None)
        import config_project as cfp

        PROJECT_DIR = cwd_project
        RESULTS_DIR = Path(getattr(cfp, "cwd_results", PROJECT_DIR / "results"))
        DATA_DIR = Path(getattr(cfp, "cwd_data", PROJECT_DIR / "data"))
        QC_DIR = Path(getattr(cfp, "cwd_data_qc", PROJECT_DIR / "data" / "20_quality_control"))
        QC_RESULTS_DIR = Path(cfp.cwd_results_qc)
        year_span = f"{pd.to_datetime(cfp.first_date, dayfirst=True).year}-{pd.to_datetime(cfp.last_date, dayfirst=True).year}"
    except Exception as e:
        print("Config import failed; falling back to manual paths. Reason:", repr(e))
        USE_CONFIG_FILES = False

if not USE_CONFIG_FILES:
    PROJECT_DIR = Path(PROJECT_DIR)
    RESULTS_DIR = PROJECT_DIR / "results"
    DATA_DIR = PROJECT_DIR / "data"
    QC_DIR = DATA_DIR / "20_quality_control"
    QC_RESULTS_DIR = RESULTS_DIR / "20_quality_control"
    year_span = "2021-2021"

benchmark_dir = QC_RESULTS_DIR / "qc_benchmark"
reference_dir = benchmark_dir / "ows_reference"
time_residual_risk_base = benchmark_dir / "residual_risk_audited"
reviewer_residual_risk_base = benchmark_dir / "residual_risk_reviewer_station_holdout"
time_fusion_base = benchmark_dir / "qc_risk_fusion_audited"
reviewer_fusion_base = benchmark_dir / "qc_risk_fusion_reviewer_station_holdout"
reviewer_output_dir = benchmark_dir / "reviewer_robustness_tests"

for p in [reference_dir, time_residual_risk_base, reviewer_residual_risk_base, time_fusion_base, reviewer_fusion_base, reviewer_output_dir]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("QC_DIR:", QC_DIR)
print("benchmark_dir:", benchmark_dir)
print("reviewer_output_dir:", reviewer_output_dir)


## 2. Import reviewer robustness script

Keep `qc_cws_reviewer_robustness_tests.py`, `qc_cws_residual_risk_model_audited.py`, and `qc_risk_fusion_audited.py` in the same scripts folder used by the previous notebooks.


In [ ]:

if USE_CONFIG_FILES and "cfm" in globals():
    SCRIPT_DIR = Path(cfm.cwd_scripts_preprocesing)
else:
    SCRIPT_DIR = Path.cwd()
    if not (SCRIPT_DIR / "qc_cws_reviewer_robustness_tests.py").exists():
        SCRIPT_DIR = Path.cwd().parent

if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

import importlib
import qc_cws_reviewer_robustness_tests as reviewer
reviewer = importlib.reload(reviewer)


rrm, qrf = reviewer.import_pipeline_modules(script_dir=SCRIPT_DIR)
print("reviewer:", reviewer.__file__)
print("residual risk script:", rrm.__file__)
print("fusion script:", qrf.__file__)

fusion_compat = reviewer.fusion_compatibility_report(qrf=qrf)
print("\nLocal fusion-script compatibility report")
print(json.dumps(fusion_compat, indent=2))
print("Reviewer script compatibility shims are active: unsupported FusionConfig kwargs are dropped automatically, and missing bias-correction/recommendation columns are derived when needed.")


## 2b. Fusion-module compatibility check

This is optional, but useful when your local `qc_risk_fusion_audited.py` is slightly older or newer than the version used to design the reviewer notebook. The reviewer script now auto-detects supported `FusionConfig` fields and back-fills key boolean columns such as `recommended_bias_correctable` when they are missing in older fusion outputs.


In [ ]:
reference_run_label = "catboost_corrected_ows_metadata_iteration01"
reference_method = "catboost"
calibration_mode = "time_train"
risk_model_key = "context_static_met"

reference_run_dir = reference_dir / reference_run_label


time_residual_risk_run_label = f"{reference_run_label}__residual_risk_AUDITED__{reference_method}__{calibration_mode}"
TIME_RESIDUAL_RISK_DIR = time_residual_risk_base / time_residual_risk_run_label
TIME_RESIDUAL_RISK_MANIFEST = TIME_RESIDUAL_RISK_DIR / f"{city}_residual_risk_manifest.json"


time_fusion_run_label = f"{time_residual_risk_run_label}__qc_risk_fusion_AUDITED__{risk_model_key}"
TIME_FUSION_DIR = time_fusion_base / time_fusion_run_label
TIME_FUSION_MANIFEST = TIME_FUSION_DIR / f"{city}_qc_risk_fusion_manifest.json"

city_stems = []
for stem in [city, city.replace("_new", ""), city.split("_")[0]]:
    stem = str(stem).strip()
    if stem and stem not in city_stems:
        city_stems.append(stem)

reference_manifest = None
reference_manifest_file = None
CWS_REFERENCE_PATH = None


if TIME_RESIDUAL_RISK_MANIFEST.exists():
    with open(TIME_RESIDUAL_RISK_MANIFEST, "r") as f:
        rr_manifest = json.load(f)

    rr_input_path = rr_manifest.get("input_path")
    if rr_input_path:
        rr_input_path = Path(rr_input_path)
        if rr_input_path.exists():
            CWS_REFERENCE_PATH = rr_input_path

    rr_source_manifest = rr_manifest.get("source_reference_manifest_path")
    if rr_source_manifest:
        rr_source_manifest = Path(rr_source_manifest)
        if rr_source_manifest.exists():
            reference_manifest_file = rr_source_manifest
            with open(reference_manifest_file, "r") as f:
                reference_manifest = json.load(f)

    print("Loaded upstream reference info from residual-risk manifest:", TIME_RESIDUAL_RISK_MANIFEST)


if CWS_REFERENCE_PATH is None:
    for stem in city_stems:
        candidate_manifest = reference_run_dir / f"{stem}_ows_reference_manifest_{reference_method}_{calibration_mode}.json"
        if not candidate_manifest.exists():
            continue
        reference_manifest_file = candidate_manifest
        with open(reference_manifest_file, "r") as f:
            reference_manifest = json.load(f)
        cws_paths = reference_manifest.get("cws_reference_paths_by_calibration_mode", {})
        candidate_path = cws_paths.get(calibration_mode) or reference_manifest.get("cws_primary_reference_path")
        if candidate_path:
            candidate_path = Path(candidate_path)
            if candidate_path.exists():
                CWS_REFERENCE_PATH = candidate_path
                break


if CWS_REFERENCE_PATH is None:
    for stem in city_stems:
        candidate_path = reference_run_dir / f"{stem}_cws_reference_residual_features_{reference_method}_{calibration_mode}.parquet"
        if candidate_path.exists():
            CWS_REFERENCE_PATH = candidate_path
            break

if CWS_REFERENCE_PATH is None:
    CWS_REFERENCE_PATH = reference_run_dir / f"{city_stems[0]}_cws_reference_residual_features_{reference_method}_{calibration_mode}.parquet"

print("reference_manifest_file:", reference_manifest_file)
print("CWS_REFERENCE_PATH:", CWS_REFERENCE_PATH)
print("exists:", CWS_REFERENCE_PATH.exists())
print("TIME_RESIDUAL_RISK_MANIFEST:", TIME_RESIDUAL_RISK_MANIFEST, TIME_RESIDUAL_RISK_MANIFEST.exists())
print("TIME_FUSION_MANIFEST:", TIME_FUSION_MANIFEST, TIME_FUSION_MANIFEST.exists())

if not CWS_REFERENCE_PATH.exists():
    raise FileNotFoundError(
        "Could not resolve the upstream CWS reference parquet. Check reference_run_label/reference_method/calibration_mode or set CWS_REFERENCE_PATH manually."
    )

if not TIME_FUSION_MANIFEST.exists():
    raise FileNotFoundError(
        f"Static fusion manifest not found:\n{TIME_FUSION_MANIFEST}\n\n"
        "Make sure notebook was run with risk_model_key='context_static_met'."
    )


## 3. Select upstream OWS-reference, residual-risk, and fusion runs

These defaults mirror the previous notebooks. Update `reference_run_label`, `reference_method`, `calibration_mode`, or the fallback paths if you use a different upstream run.


In [ ]:
RUN_STATION_HOLDOUT = False
RUN_FUSION_ON_STATION_HOLDOUT = False
RUN_BOOTSTRAP_EXISTING_FUSION = True
RUN_STATION_DIAGNOSTICS_EXISTING_FUSION = True
RUN_SHIFT_DIAGNOSTICS_EXISTING_FUSION = True

STATION_HOLDOUT_SEEDS = [11, 22, 33, 44, 55]
FEATURE_MODES = "context_history,context_static_met"
VALID_FRAC = 0.15
TEST_FRAC = 0.15


BOOTSTRAP_UNIT = "station"
N_BOOTSTRAP = 500


BOOTSTRAP_METHODS_REGEX = None


FAST_DEBUG_MODE = False
if FAST_DEBUG_MODE:
    ITERATIONS = 200
    MAX_TRAIN_ROWS = 500_000
    MAX_VALID_ROWS = 200_000
    VERBOSE = 50
    N_BOOTSTRAP = 100
    STATION_HOLDOUT_SEEDS = [11]
else:
    ITERATIONS = 1200
    MAX_TRAIN_ROWS = None
    MAX_VALID_ROWS = None
    VERBOSE = 100

reviewer_output_dir = benchmark_dir / f"reviewer_robustness_tests__{risk_model_key}"
reviewer_output_dir.mkdir(parents=True, exist_ok=True)

cfg = reviewer.ReviewerRobustnessConfig(
    city=city,
    cws_reference_path=str(CWS_REFERENCE_PATH),
    source_reference_manifest_path=str(reference_manifest_file) if (reference_manifest_file is not None and Path(reference_manifest_file).exists()) else None,
    source_reference_run_label=reference_run_label,
    reference_method=reference_method,
    calibration_mode=calibration_mode,

    time_residual_risk_manifest=str(TIME_RESIDUAL_RISK_MANIFEST) if TIME_RESIDUAL_RISK_MANIFEST.exists() else None,
    time_fusion_manifest=str(TIME_FUSION_MANIFEST) if TIME_FUSION_MANIFEST.exists() else None,
    time_fused_path=None,

    output_dir=str(reviewer_output_dir),
    residual_risk_output_dir=str(reviewer_residual_risk_base),
    fusion_output_dir=str(reviewer_fusion_base),

    run_station_holdout=RUN_STATION_HOLDOUT,
    run_fusion_on_station_holdout=RUN_FUSION_ON_STATION_HOLDOUT,
    run_bootstrap_existing_fusion=RUN_BOOTSTRAP_EXISTING_FUSION,
    run_station_diagnostics_existing_fusion=RUN_STATION_DIAGNOSTICS_EXISTING_FUSION,
    run_shift_diagnostics_existing_fusion=RUN_SHIFT_DIAGNOSTICS_EXISTING_FUSION,

    station_holdout_seeds=tuple(STATION_HOLDOUT_SEEDS),
    valid_frac=VALID_FRAC,
    test_frac=TEST_FRAC,
    target_mode="auto",
    feature_modes=FEATURE_MODES,

    iterations=ITERATIONS,
    learning_rate=0.04,
    depth=8,
    l2_leaf_reg=8.0,
    auto_class_weights="Balanced",
    early_stopping_rounds=100,
    thread_count=None,
    used_ram_limit="42gb",
    use_gpu=False,
    verbose=VERBOSE,
    max_train_rows=MAX_TRAIN_ROWS,
    max_valid_rows=MAX_VALID_ROWS,

    fusion_risk_model_keys=(risk_model_key,),
    qc_dir=str(QC_DIR) if QC_DIR.exists() else None,
    qc_lenient=None,
    qc_strict=None,
    qc_ultra_strict=None,
    fusion_overwrite=True,
    fusion_use_reliability_calibration=True,

    bootstrap_unit=BOOTSTRAP_UNIT,
    n_bootstrap=N_BOOTSTRAP,
    bootstrap_random_state=2026,
    bootstrap_methods_regex=BOOTSTRAP_METHODS_REGEX,
)

print(cfg)


## 4. User-editable reviewer-test configuration

Recommended paper run:

- `RUN_STATION_HOLDOUT = True`
- `STATION_HOLDOUT_SEEDS = [11, 22, 33, 44, 55]`
- `FEATURE_MODES = "context_history,context_static_met"`
- `RUN_FUSION_ON_STATION_HOLDOUT = False` initially, then switch on after the station-held-out risk runs are complete if you want cold-start fusion tables.
- `N_BOOTSTRAP = 500` for final tables; use 100 for a quick check.

The reviewer script is now backward-compatible with older `qc_risk_fusion_audited.py` versions:

- unsupported `FusionConfig` kwargs are dropped automatically;
- missing reviewer columns such as `recommended_bias_correctable` are inferred from `fusion_category`, `recommended_action`, and available correction fields;
- if needed, reviewer-side method/correction tables are rebuilt from the fused parquet.

For a quick dry run, set `FAST_DEBUG_MODE = True`.


In [ ]:
manifest = reviewer.run_reviewer_robustness_suite(cfg, script_dir=SCRIPT_DIR)
print("Fusion compatibility used by reviewer suite")
print(json.dumps(manifest.get("fusion_compatibility", {}), indent=2))
print("\nReviewer output paths")
print(json.dumps(manifest["outputs"], indent=2))


## 5. Run reviewer robustness suite

This will write all summaries under `reviewer_output_dir`. The station-held-out CatBoost runs are the only computationally heavy part. Bootstrap/station diagnostics are much lighter once the fused file exists.


In [ ]:
manifest = reviewer.run_reviewer_robustness_suite(cfg, script_dir=SCRIPT_DIR)
print(json.dumps(manifest["outputs"], indent=2))


In [ ]:



fused_path, fusion_manifest = reviewer.resolve_fused_path_from_manifest(
    TIME_FUSION_MANIFEST,
    direct_path=None,
)

if fused_path is None or not Path(fused_path).exists():
    raise FileNotFoundError(f"Could not resolve fused_path from: {TIME_FUSION_MANIFEST}")

fused = reviewer.load_dataframe_auto(fused_path)
fused, notes = reviewer.harmonize_fusion_output_schema(fused)

if notes:
    print("Harmonized fused output:", notes)

valid = fused[fused["split"].astype(str).eq("valid")].copy()
test = fused[fused["split"].astype(str).eq("test")].copy()

del fused
gc.collect()

cal_prob_col = reviewer.infer_calibrated_prob_col(
    test,
    manifest=fusion_manifest,
    risk_model_key=risk_model_key,
)

print("Fused path:", fused_path)
print("Validation rows:", len(valid))
print("Test rows:", len(test))
print("Calibrated probability column:", cal_prob_col)


In [ ]:



sample_cols = [
    "date", "network", "station_id",
    "temp_raw", "ref_mu", "ref_sigma",
    "cws_ref_resid", "abs_cws_ref_resid",
    "cws_ref_z", "abs_cws_ref_z",
    "cws_ref_resid_corrected_station_bias_policy",
    "fusion_category", "recommended_action", "fusion_reason",

    "p_reference_model_issue_no_qc",
    "p_any_quality_issue",
    "p_reject_as_unexplained_transient_error",
    "p_residual_explained_by_persistent_station_context",
    "p_radiation_or_siting_bias_signal",
    "p_reference_uncertainty",
    "p_context_mismatch_support",
    "p_context_directional_microclimate_support",
    "p_environmental_difference_signal",
    "p_context_microclimate_support",
    "p_qc_crowdqc", "qc_flag_pattern",

    "expected_train_station_context_bias_c",
    "station_bias_correction_c",
    "analysis_weight",


    "is_daylight", "fusion_is_high_solar", "fusion_is_low_wind",
    "ssrd_wm2", "wind_speed", "wind_speed_for_radiation_ms",
    "tp_mm", "is_rain",


    "ref_support_score", "ref_support_class",
    "ref_local_n", "ref_nearest_dist_km",
    "ref_nearest_value_dist_km", "ref_local_spread_c",


    "LC_point_lg", "LC_buffer_lg",
    "LCZ_point_lg", "LCZ_buffer_lg",
    "LC_buffer_fraction", "LCZ_buffer_fraction",
    "building_height_m", "elev_meters",


    "NDVI", "LST_C", "LST_C_asof",
    "LST_C_gapfill_3h", "LST_C_gapfill_clim",
    "ndvi_minus_ows_idw_ndvi",
    "ndvi_minus_ows_nearest_ndvi",
    "lst_c_gapfill_clim_minus_ows_idw_lst",
    "lst_gapfill_clim_minus_ows_idw_lst",
    "lst_c_asof_minus_ows_idw_lst",
    "lst_asof_minus_ows_idw_lst",
    "cws_minus_ows_idw_elev_meters",
    "cws_minus_ows_idw_building_height_m",
]

if cal_prob_col and cal_prob_col not in sample_cols:
    sample_cols.append(cal_prob_col)

sample_cols = [c for c in sample_cols if c in test.columns]

sample_dir = reviewer_output_dir / "enriched_candidate_samples"
sample_dir.mkdir(parents=True, exist_ok=True)

enriched_paths = {}
for category, g in test.groupby("fusion_category", dropna=False):
    n = min(250, len(g))
    if n <= 0:
        continue

    sample = g.sample(n=n, random_state=42) if len(g) > n else g.copy()
    safe = "".join(ch if ch.isalnum() or ch in "-_." else "_" for ch in str(category))
    path = sample_dir / f"sample__{safe}.csv"
    sample[sample_cols].to_csv(path, index=False)
    enriched_paths[str(category)] = str(path)

pd.DataFrame(
    [{"fusion_category": k, "path": v} for k, v in enriched_paths.items()]
).to_csv(sample_dir / "enriched_candidate_sample_manifest.csv", index=False)

print("Saved enriched samples to:", sample_dir)
print("Sample columns:", len(sample_cols))


In [ ]:



score_cols = {
    "fusion_reject_score": "p_reject_as_unexplained_transient_error",
}

if cal_prob_col:
    score_cols[f"risk_prob_{risk_model_key}"] = cal_prob_col


def residual_metrics(df, mask, residual_col="cws_ref_resid"):
    mask = pd.Series(mask, index=df.index).fillna(False).astype(bool)
    g = df.loc[mask]
    r = pd.to_numeric(g[residual_col], errors="coerce").dropna()
    abs_r = r.abs()

    return {
        "n_total": int(len(df)),
        "n_kept": int(mask.sum()),
        "retention": float(mask.mean()),
        "n_stations_kept": int(g[["network", "station_id"]].drop_duplicates().shape[0]),
        "mae": float(abs_r.mean()) if len(abs_r) else np.nan,
        "rmse": float(np.sqrt((r ** 2).mean())) if len(r) else np.nan,
        "p95_abs": float(abs_r.quantile(0.95)) if len(abs_r) else np.nan,
        "p99_abs": float(abs_r.quantile(0.99)) if len(abs_r) else np.nan,
    }


rows = []

for level in ["lenient", "strict", "ultra_strict"]:
    qc_col = f"qc_{level}_is_outlier"
    if qc_col not in valid.columns or qc_col not in test.columns:
        print(f"Skipping {level}; missing {qc_col}")
        continue

    valid_qc_clean = pd.to_numeric(valid[qc_col], errors="coerce").fillna(0).eq(0)
    test_qc_clean = pd.to_numeric(test[qc_col], errors="coerce").fillna(0).eq(0)

    target_retention = float(valid_qc_clean.mean())

    row = {
        "target_from": f"crowdqc_{level}_valid_clean_retention",
        "method": f"crowdqc_{level}_clean_test",
        "score_col": qc_col,
        "threshold_from_valid": np.nan,
        "target_retention_valid": target_retention,
        "score_nonmissing_valid": float(valid[qc_col].notna().mean()),
        "score_nonmissing_test": float(test[qc_col].notna().mean()),
        "residual_col": "cws_ref_resid",
    }
    row.update(residual_metrics(test, test_qc_clean, "cws_ref_resid"))
    rows.append(row)

    for method_name, score_col in score_cols.items():
        if score_col not in valid.columns or score_col not in test.columns:
            print(f"Skipping {method_name}; missing {score_col}")
            continue

        score_valid = pd.to_numeric(valid[score_col], errors="coerce")
        score_test = pd.to_numeric(test[score_col], errors="coerce")

        threshold = float(score_valid.quantile(target_retention))
        keep_test = score_test <= threshold

        row = {
            "target_from": f"crowdqc_{level}_valid_clean_retention",
            "method": f"{method_name}_matched_to_{level}",
            "score_col": score_col,
            "threshold_from_valid": threshold,
            "target_retention_valid": target_retention,
            "score_nonmissing_valid": float(score_valid.notna().mean()),
            "score_nonmissing_test": float(score_test.notna().mean()),
            "residual_col": "cws_ref_resid",
        }
        row.update(residual_metrics(test, keep_test, "cws_ref_resid"))
        rows.append(row)

        corrected_col = "cws_ref_resid_corrected_station_bias_policy"
        if corrected_col in test.columns:
            row = {
                "target_from": f"crowdqc_{level}_valid_clean_retention",
                "method": f"{method_name}_matched_to_{level}_policy_bias_corrected",
                "score_col": score_col,
                "threshold_from_valid": threshold,
                "target_retention_valid": target_retention,
                "score_nonmissing_valid": float(score_valid.notna().mean()),
                "score_nonmissing_test": float(score_test.notna().mean()),
                "residual_col": corrected_col,
            }
            row.update(residual_metrics(test, keep_test, corrected_col))
            rows.append(row)

matched_frontier = pd.DataFrame(rows)

matched_path = reviewer_output_dir / f"{city}_static_fusion_validation_matched_retention_frontier.csv"
matched_frontier.to_csv(matched_path, index=False)

display(matched_frontier)
print("Saved:", matched_path)


In [ ]:
tail_resid_col = (
    "cws_ref_resid_corrected_station_bias_policy"
    if "cws_ref_resid_corrected_station_bias_policy" in test.columns
    else "cws_ref_resid"
)

test["abs_tail_resid"] = pd.to_numeric(test[tail_resid_col], errors="coerce").abs()

tail_mask = (
    reviewer._series_bool(test, "recommended_keep_microclimate")
    & test["abs_tail_resid"].ge(3.0)
)

tail = test.loc[tail_mask].copy()

group_cols = [
    "fusion_category",
    "recommended_action",
    "network",
    "is_daylight",
    "fusion_is_high_solar",
    "fusion_is_low_wind",
    "ref_support_class",
    "qc_flag_pattern",
]

tail_rows = []
for col in group_cols:
    if col not in tail.columns:
        continue
    for key, g in tail.groupby(col, dropna=False):
        r = pd.to_numeric(g[tail_resid_col], errors="coerce")
        tail_rows.append({
            "group_col": col,
            "group_value": str(key),
            "n_tail_rows": int(len(g)),
            "tail_fraction": float(len(g) / max(len(tail), 1)),
            "mean_abs_resid": float(r.abs().mean()),
            "p95_abs_resid": float(r.abs().quantile(0.95)),
            "n_stations": int(g[["network", "station_id"]].drop_duplicates().shape[0]),
        })

tail_audit = pd.DataFrame(tail_rows).sort_values(
    ["group_col", "n_tail_rows"],
    ascending=[True, False],
)

tail_path = reviewer_output_dir / f"{city}_static_fusion_microclimate_keep_tail_audit_abs_ge_3c.csv"
tail_audit.to_csv(tail_path, index=False)

display(tail_audit)
print("Saved:", tail_path)


## 6. Load and inspect reviewer outputs

These are the tables I would use directly in the paper or appendix.


In [ ]:
outputs = manifest["outputs"]

def read_output(key):
    p = outputs.get(key)
    if not p:
        print("Missing output key:", key)
        return pd.DataFrame()
    p = Path(p)
    if not p.exists():
        print("Output path not found:", p)
        return pd.DataFrame()
    return pd.read_csv(p)

station_holdout_metrics = read_output("station_holdout_metrics_path")
station_holdout_metric_summary = read_output("station_holdout_metric_summary_path")
station_holdout_feature_stability = read_output("station_holdout_feature_importance_stability_path")
policy_bootstrap_ci = read_output("existing_fusion_policy_bootstrap_ci_path")
correction_bootstrap_ci = read_output("existing_fusion_correction_bootstrap_ci_path")
category_bootstrap_ci = read_output("existing_fusion_category_bootstrap_ci_path")
action_bootstrap_ci = read_output("existing_fusion_action_bootstrap_ci_path")
station_dominance = read_output("dominance_summary_path")
station_summary = read_output("station_summary_path")
shift_by_split = read_output("shift_by_split_path")
shift_by_split_month = read_output("shift_by_split_month_path")

print("Station-holdout metric summary")
display(station_holdout_metric_summary)

print("Feature-importance stability, top rows")
display(station_holdout_feature_stability.head(40))

print("Policy bootstrap CIs")
display(policy_bootstrap_ci)

print("Correction bootstrap CIs")
display(correction_bootstrap_ci)

print("Category fraction bootstrap CIs")
display(category_bootstrap_ci.head(20))

print("Station dominance summary")
display(station_dominance)

print("Top station diagnostics")
display(station_summary.head(30))

print("Shift by split")
display(shift_by_split)


## 7. Compare existing temporal split against new CWS station-held-out split

This table is important for the paper because it separates two claims:

- **Operational known-station monitoring:** the existing temporal split.
- **Cold-start unseen-station generalization:** the new station-held-out split.

It is acceptable if station-held-out performance is lower; that is exactly the reviewer question the test answers.


In [ ]:
comparison_parts = []

if TIME_RESIDUAL_RISK_MANIFEST.exists():
    time_manifest = json.load(open(TIME_RESIDUAL_RISK_MANIFEST, "r"))
    time_metrics_path = Path(time_manifest.get("metrics_path", ""))
    if time_metrics_path.exists():
        time_metrics = pd.read_csv(time_metrics_path)
        time_test = time_metrics[time_metrics["split"].astype(str).eq("test")].copy()
        time_test.insert(0, "evaluation", "temporal_known_station_test")
        comparison_parts.append(time_test)

if len(station_holdout_metrics):
    sho_test = station_holdout_metrics[station_holdout_metrics["split"].astype(str).eq("test")].copy()
    sho_summary = (
        sho_test.groupby("model", as_index=False)
        .agg(
            n_runs=("station_holdout_seed", "nunique"),
            auroc_mean=("auroc", "mean"),
            auroc_std=("auroc", "std"),
            auprc_mean=("auprc", "mean"),
            auprc_std=("auprc", "std"),
            f1_mean=("f1", "mean"),
            f1_std=("f1", "std"),
            brier_mean=("brier", "mean"),
            brier_std=("brier", "std"),
            ece_15_mean=("ece_15", "mean"),
            ece_15_std=("ece_15", "std"),
            event_rate_mean=("event_rate", "mean"),
            event_rate_std=("event_rate", "std"),
        )
    )
    print("Station-held-out summary over seeds:")
    display(sho_summary)

if comparison_parts:
    known_station_test = pd.concat(comparison_parts, ignore_index=True)
    keep_cols = [c for c in ["evaluation", "model", "split", "n", "event_rate", "auroc", "auprc", "f1", "brier", "ece_15", "threshold"] if c in known_station_test.columns]
    display(known_station_test[keep_cols])


## 8. Reviewer-facing plots

These plots are designed to be directly interpretable by collaborators/reviewers.


In [ ]:

if len(station_holdout_metrics):
    plot_df = station_holdout_metrics[station_holdout_metrics["split"].astype(str).eq("test")].copy()
    for metric in ["auroc", "auprc", "f1", "brier", "ece_15"]:
        if metric not in plot_df.columns:
            continue
        plt.figure(figsize=(9, 4.5))
        for model, g in plot_df.groupby("model"):
            plt.plot(g["station_holdout_seed"], g[metric], marker="o", label=model)
        plt.xlabel("Station-holdout seed")
        plt.ylabel(metric)
        plt.title(f"CWS station-held-out test: {metric}")
        plt.legend()
        plt.tight_layout()
        plt.show()


if len(policy_bootstrap_ci):
    show_methods = [
        "raw_all_observed",
        "crowdqc_lenient_clean_available_only",
        "crowdqc_strict_clean_available_only",
        "risk_context_history_p_le_0.8",
        "fusion_keep_microclimate",
        "fusion_keep_conservative",
    ]
    plot_df = policy_bootstrap_ci[policy_bootstrap_ci["method"].isin(show_methods)].copy()
    if len(plot_df):
        plt.figure(figsize=(8, 5))
        x = plot_df["point_retention"]
        y = plot_df["point_residual_mae"]
        xerr = np.vstack([
            x - plot_df["retention_ci_low"],
            plot_df["retention_ci_high"] - x,
        ])
        yerr = np.vstack([
            y - plot_df["residual_mae_ci_low"],
            plot_df["residual_mae_ci_high"] - y,
        ])
        plt.errorbar(x, y, xerr=xerr, yerr=yerr, fmt="o", capsize=3)
        for _, r in plot_df.iterrows():
            plt.annotate(r["method"], (r["point_retention"], r["point_residual_mae"]), fontsize=8, alpha=0.8)
        plt.xlabel("Retention, station-bootstrap 95% CI")
        plt.ylabel("Residual MAE (°C), station-bootstrap 95% CI")
        plt.title("Policy retention vs residual error with station-level uncertainty")
        plt.tight_layout()
        plt.show()


if len(correction_bootstrap_ci):
    show_methods = [
        "raw_all_observed_no_bias_correction",
        "fusion_keep_microclimate_no_bias_correction",
        "fusion_keep_microclimate_policy_bias_corrected",
        "fusion_keep_conservative_no_bias_correction",
        "fusion_bias_correctable_subset_no_bias_correction",
        "fusion_bias_correctable_subset_policy_bias_corrected",
    ]
    plot_df = correction_bootstrap_ci[correction_bootstrap_ci["method"].isin(show_methods)].copy()
    if len(plot_df):
        plot_df = plot_df.sort_values("point_residual_mae", ascending=False)
        plt.figure(figsize=(10, 5))
        y_pos = np.arange(len(plot_df))
        x = plot_df["point_residual_mae"]
        xerr = np.vstack([
            x - plot_df["residual_mae_ci_low"],
            plot_df["residual_mae_ci_high"] - x,
        ])
        plt.barh(y_pos, x)
        plt.errorbar(x, y_pos, xerr=xerr, fmt="none", capsize=3)
        plt.yticks(y_pos, plot_df["method"])
        plt.xlabel("Residual MAE (°C), station-bootstrap 95% CI")
        plt.title("Effect of weak train-only station-bias correction")
        plt.tight_layout()
        plt.show()


if len(category_bootstrap_ci):
    plot_df = category_bootstrap_ci.head(12).copy().sort_values("point_fraction")
    plt.figure(figsize=(10, 6))
    y_pos = np.arange(len(plot_df))
    x = plot_df["point_fraction"]
    xerr = np.vstack([
        x - plot_df["fraction_ci_low"],
        plot_df["fraction_ci_high"] - x,
    ])
    plt.barh(y_pos, x)
    plt.errorbar(x, y_pos, xerr=xerr, fmt="none", capsize=3)
    plt.yticks(y_pos, plot_df["category"])
    plt.xlabel("Fraction of evaluation rows, station-bootstrap 95% CI")
    plt.title("Fusion category mix with station-level uncertainty")
    plt.tight_layout()
    plt.show()


if len(station_dominance):
    plt.figure(figsize=(7, 4.5))
    plt.plot(station_dominance["top_k_stations"], station_dominance["row_share"], marker="o", label="Row share")
    plt.plot(station_dominance["top_k_stations"], station_dominance["raw_abs_error_share"], marker="o", label="Absolute-error share")
    plt.xlabel("Top-k stations by absolute-error contribution")
    plt.ylabel("Share")
    plt.title("Do a few stations dominate residual error?")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 9. How to report these results

Use the station-held-out table to say whether the residual-risk model is robust to unseen CWS stations. Use the bootstrap tables to add confidence intervals to the main policy and correction claims. Use the station dominance table to show whether improvements are broad or concentrated in a few problematic stations.

Suggested paper language:

> We report two CWS residual-risk evaluations: a temporal known-station setting for operational monitoring, and a station-held-out cold-start setting in which all observations from a subset of CWS stations are excluded from training. This separates detection of recurring station behavior from generalization based on metadata, meteorology, satellite context, and local OWS support.

> We additionally use station-level block bootstrap intervals for policy metrics to avoid overstating improvements from millions of hourly rows that are not independent.
